# Eksplorasi AutoModelForCausalLM

**Task**: Text Generation / Autoregressive
**Cara Kerja**: Memprediksi token berikutnya dari kiri ke kanan berdasarkan token-token sebelumnya.
**Model Populer**: GPT-2, GPT-Neo, GPT-4, Llama 3, Mistral, dll.
**Dataset**: `wikitext` (WikiText-2) - dataset yang sangat populer yang berisi kumpulan artikel Wikipedia berkualitas tinggi. Sering digunakan untuk melatih / fine-tune model bahasa Causal LM.

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, set_seed
from datasets import load_dataset
import torch

## 1. Load Dataset Publik (`wikitext`)
Kita akan memuat dataset WikiText-2. Causal LM pada dasarnya hanya menerima sequence of text (teks biasa), lalu belajar memprediksi kata (token) selanjutnya. Jadi teks panjang seperti artikel wiki sangat ideal.

In [8]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# Filter baris yang kosong atau terlalu pendek agar contohnya lebih bagus

dataset = dataset.filter(lambda x: len(x['text'].strip()) > 50)

# print("Contoh teks dari dataset WikiText:\n")
print(dataset[0]['text'])

 Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . 



## 2. Load Tokenizer & Model
Kita akan memuat model `gpt2` dari Hugging Face. Model ini merupakan base model generative yang sering jadi titik awal untuk memahami text generation.

In [9]:
model_checkpoint = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForCausalLM.from_pretrained(model_checkpoint)

# GPT-2 tidak memiliki pad_token bawaan, kita samakan saja dengan eos_token (End of Sequence)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [17]:
print(model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


## 3. Text Generation (Manual dari awal)
Di sini kita mendemonstrasikan bagaimana kalimat input (prompt) diproses token menjadi angka, diberikan ke model, lalu hasilnya dikembalikan menjadi teks (*decode*).

In [19]:
prompt = "The history of artificial intelligence began with "
inputs = tokenizer(prompt, return_tensors="pt")

# Menggunakan metode generate pada model Causal LM
outputs = model.generate(
    **inputs,
    max_new_tokens=40,       # Maksimal generate 40 token baru 
    num_return_sequences=1,  # Hasilkan 1 output kalimat terpisah
    temperature=0.7,         # Mengatur tingkat kreativitas model (rendah = lebih kaku, tinggi=random)
    do_sample=True,          # Perlu True jika ingin menggunakan randomness (temperature)
    pad_token_id=tokenizer.eos_token_id
)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("--- Hasil Generator (model.generate) ---\n")
print(generated_text)

--- Hasil Generator (model.generate) ---

The history of artificial intelligence began with  the invention of  artificial intelligence in the early 1950s.   It was the first artificial intelligence program that was ever developed as part of the   Human Technology Initiative  program in


## 4. Persiapan Data untuk Training Manual
Untuk training mandiri dengan PyTorch, kita harus menyiapkan objek `DataLoader`. Pada model Causal LM, target (`labels`) adalah input teks itu sendiri; arsitektur Hugging Face secara otomatis akan menggeser rentang waktu (shift) label saat menghitung "Cross Entropy Loss" secara internal.

In [12]:
from torch.utils.data import Dataset, DataLoader

# Membatasi panjang token agar memori tidak penuh dan proses cepat untuk contoh
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128, padding="max_length")

# Ambil sampel kecil data (misal 50 baris) agar simulasi training berjalan cepat
train_sample = dataset.select(range(50))
tokenized_train = train_sample.map(tokenize_function, batched=True, remove_columns=['text'])

class CausalLMDataset(Dataset):
    def __init__(self, token_data):
        self.data = token_data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        input_ids = torch.tensor(item['input_ids'])
        attention_mask = torch.tensor(item['attention_mask'])
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            # Supaya model CausalLM (GPT-2) dapat menghitung loss, labels = input_ids
            "labels": input_ids.clone() 
        }

train_dataset = CausalLMDataset(tokenized_train)
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True)
print(f"Total Batch: {len(train_dataloader)}")
print("Selesai menyiapkan PyTorch DataLoader!")

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Total Batch: 13
Selesai menyiapkan PyTorch DataLoader!


In [16]:
for data in train_dataloader:
    print(data.keys())
    break

dict_keys(['input_ids', 'attention_mask', 'labels'])


## 5. Training dan Evaluasi Loop (PyTorch Manual)
Kita gunakan `torch.optim.AdamW`. Dalam tahapan iterasi *Training*:
1. `Forward Pass`: mengirim tensor menuju model untuk dapatkan prediksi dan memunculkan error loss.
2. `Backward Pass`: komputasi backpropagation mengkalkulasi gradien dari eror loss.
3. `Optimizer Step`: update parameter bobot model.

Setelah training, dilakukan mode `model.eval()`. Pada Causal LM metrik pembanding kinerjanya sangat umum disuguhkan dalam angka **Perplexity** (Eksponensial dari log loss). Makin kecil angka perplexity makin baik LLM tersebut memahami bahasa.

In [20]:
from torch.optim import AdamW
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)

epochs = 1
print("==== Memulai Training ====")
model.train()
for epoch in range(epochs):
    total_train_loss = 0
    for step, batch in enumerate(train_dataloader):
        optimizer.zero_grad()
        
        # Pindahkan tensor input ke memori device (GPU/CPU)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # Forward pass (Secara internal akan melakukan shift index logits pada fungsi Loss-nya)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        # Backward & Update bobot
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        
        if step % 5 == 0:
            print(f"Epoch {epoch+1} | Step {step} | Loss {loss.item():.4f}")
            
    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f">> Rata-rata Train Loss Epoch {epoch+1}: {avg_train_loss:.4f}\n")

==== Memulai Training ====


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 1 | Step 0 | Loss 6.3851
Epoch 1 | Step 5 | Loss 2.0234
Epoch 1 | Step 10 | Loss 3.1615
>> Rata-rata Train Loss Epoch 1: 3.6192



In [21]:
# ================================
# Evaluasi pada data Validation
# ================================
print("==== Memulai Evaluasi ====")
# Simulasikan data validasi kecil
val_sample = dataset.select(range(50, 70))
tokenized_val = val_sample.map(tokenize_function, batched=True, remove_columns=['text'])
val_dataloader = DataLoader(CausalLMDataset(tokenized_val), batch_size=4)

model.eval()
total_val_loss = 0
with torch.no_grad():
    for batch in val_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_val_loss += outputs.loss.item()

avg_val_loss = total_val_loss / len(val_dataloader)
perplexity = math.exp(avg_val_loss)

print(f"Validation Loss : {avg_val_loss:.4f}")
print(f"Perplexity      : {perplexity:.4f}")

==== Memulai Evaluasi ====


Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Validation Loss : 2.7530
Perplexity      : 15.6895


## 6. Inferensi Hasil Akhir Sesudah Training (Text Generation)
Metode pada `model.generate` yang kita jalankan sekarang, secara otomatis akan memanggil bobot neural-network parameter yang telah diperbarui akibat proses *fine-tuning* pada loop iterasi PyTorch di atas.

In [25]:
prompt = "The history of artificial intelligence began with "
# Pastikan tensor prompt juga menempati GPU jika proses training memakai GPU
inputs = tokenizer(prompt, return_tensors="pt").to(device)

model.eval()
outputs = model.generate(
    **inputs,
    max_new_tokens=40,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("--- Hasil Inferensi Generator Setelah Training ---\n")
print(generated_text)

--- Hasil Inferensi Generator Setelah Training ---

The history of artificial intelligence began with  " The Turing Test ," which was designed to simulate the brain's ability to recognize complex patterns and tasks as well as learn from them. After more than 100 years of research , researchers began to develop
